# Model Comparison: 6-Model Benchmark

Trains and compares Logistic Regression, Random Forest, XGBoost, CatBoost, LightGBM, and Neural Net on the engineered feature matrix.
Uses Optuna for hyperparameter tuning and reports AUC-ROC, AUC-PR, F1, and calibration.

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score

FEATURES_PATH = Path('../outputs/features_matrix.parquet')
LABELS_PATH = Path('../data/train_labels.csv')
MODELS_DIR = Path('../outputs/models')
PLOTS_DIR = Path('../outputs/plots')
MODELS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

N_OPTUNA_TRIALS = 20  # Increase to 100+ for production
RANDOM_STATE = 42
print('Setup complete')

## 1. Data Loading

In [ ]:
if not FEATURES_PATH.exists():
    raise FileNotFoundError(f'Run the feature pipeline first: {FEATURES_PATH}')

X = pd.read_parquet(FEATURES_PATH)
labels_df = pd.read_csv(LABELS_PATH, index_col='account_id')
y = labels_df.loc[X.index, 'is_mule']

print(f'Feature matrix: {X.shape}')
print(f'Mule rate: {y.mean():.3%}')
X.head()

## 2. Train/Validation Split (Stratified 80/20)

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print(f'Train: {X_train.shape}, mule rate: {y_train.mean():.3%}')
print(f'Val:   {X_val.shape}, mule rate: {y_val.mean():.3%}')

## 3. Train All 6 Models with Optuna

In [ ]:
from src.models.trainer import ModelTrainer

trainer = ModelTrainer(output_dir=str(MODELS_DIR))
all_results = trainer.train_all_models(
    X_train, y_train, X_val, y_val, n_trials=N_OPTUNA_TRIALS
)

print('\nTraining complete. Val AUC-ROC:')
for name, res in all_results.items():
    print(f'  {name:25s}: {res["best_score"]:.4f}')

## 4. Generate Full Evaluation Metrics

In [ ]:
from src.models.evaluator import ModelEvaluator

evaluator = ModelEvaluator(plots_dir=str(PLOTS_DIR))

eval_data = {}
for name, res in all_results.items():
    wrapper = res['model']
    y_prob = wrapper.predict_proba(X_val)
    metrics = evaluator.evaluate(y_val, y_prob)
    eval_data[name] = {
        'y_true': y_val.values,
        'y_prob': y_prob,
        'threshold': metrics['threshold'],
        'metrics': metrics,
        'model': wrapper,
    }

print('Evaluation complete')

## 5. Model Comparison Table

In [ ]:
comparison_df = evaluator.compare_models(eval_data)
display_cols = ['auc_roc', 'auc_pr', 'f1', 'precision', 'recall', 'brier']
comparison_df[display_cols].round(4)

## 6. Overlaid ROC Curves

In [ ]:
try:
    import plotly.graph_objects as go
    from sklearn.metrics import roc_curve

    fig = go.Figure()
    colors = ['#636EFA','#EF553B','#00CC96','#AB63FA','#FFA15A','#19D3F3']
    for (name, res), color in zip(eval_data.items(), colors):
        fpr, tpr, _ = roc_curve(res['y_true'], res['y_prob'])
        auc = res['metrics']['auc_roc']
        fig.add_trace(go.Scatter(x=fpr, y=tpr, name=f'{name} (AUC={auc:.3f})', line=dict(color=color)))

    fig.add_trace(go.Scatter(x=[0,1], y=[0,1], name='Random', line=dict(dash='dash', color='grey')))
    fig.update_layout(title='ROC Curves — All Models', xaxis_title='FPR', yaxis_title='TPR',
                      width=700, height=500)
    fig.show()
    fig.write_html(str(PLOTS_DIR / 'roc_curves_interactive.html'))
except ImportError:
    evaluator.plot_roc_curves(eval_data)
    print('Saved static ROC curves (plotly not available)')

## 7. Overlaid Precision-Recall Curves

In [ ]:
try:
    import plotly.graph_objects as go
    from sklearn.metrics import precision_recall_curve, average_precision_score

    fig = go.Figure()
    for (name, res), color in zip(eval_data.items(), colors):
        prec, rec, _ = precision_recall_curve(res['y_true'], res['y_prob'])
        ap = res['metrics']['auc_pr']
        fig.add_trace(go.Scatter(x=rec, y=prec, name=f'{name} (AP={ap:.3f})', line=dict(color=color)))

    fig.update_layout(title='Precision-Recall Curves — All Models',
                      xaxis_title='Recall', yaxis_title='Precision', width=700, height=500)
    fig.show()
    fig.write_html(str(PLOTS_DIR / 'pr_curves_interactive.html'))
except ImportError:
    evaluator.plot_pr_curves(eval_data)
    print('Saved static PR curves')

## 8. Confusion Matrices (2×3 Grid)

In [ ]:
evaluator.plot_confusion_matrices(eval_data, save_name='confusion_matrices.png')
from IPython.display import Image
Image(str(PLOTS_DIR / 'confusion_matrices.png'))

## 9. Calibration Plots (Top 3 Models)

In [ ]:
evaluator.plot_calibration(eval_data, top_n=3, save_name='calibration.png')
Image(str(PLOTS_DIR / 'calibration.png'))

## 10. Statistical Significance (5-Fold CV + Wilcoxon)

In [ ]:
from src.models.selector import ModelSelector
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_results = {}
for name, res in eval_data.items():
    wrapper = res['model']
    if hasattr(wrapper.model, 'predict_proba'):
        scores = cross_val_score(wrapper.model, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)
        cv_results[name] = {'cv_scores': scores.tolist(), 'metrics': res['metrics']}
        print(f'{name:25s}: CV AUC = {scores.mean():.4f} ± {scores.std():.4f}')

selector = ModelSelector()
if len(cv_results) > 1:
    pval_matrix = selector.statistical_comparison(cv_results)
    print('\nWilcoxon p-values (pairwise):')
    display(pval_matrix.round(4))

## 11. Feature Importance — Top 20 (Best Model)

In [ ]:
import matplotlib.pyplot as plt

best_name, best_res = selector.select_best(eval_data)
print(f'Best model: {best_name} (AUC-ROC: {best_res["metrics"]["auc_roc"]:.4f})')

best_wrapper = best_res['model']
fi = best_wrapper.get_feature_importance()

if fi and 'importances' in fi:
    importances = fi['importances']
    feat_names = list(X.columns)
    top20_idx = np.argsort(importances)[::-1][:20]
    
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.barh([feat_names[i] for i in top20_idx[::-1]], importances[top20_idx[::-1]])
    ax.set_xlabel('Feature Importance')
    ax.set_title(f'Top 20 Features — {best_name}')
    plt.tight_layout()
    plt.savefig(str(PLOTS_DIR / 'feature_importance_top20.png'), dpi=150)
    plt.show()
else:
    print(f'Feature importance not available for {best_name}')

## 12. Learning Curves (Best Model)

In [ ]:
from sklearn.model_selection import learning_curve

best_wrapper = best_res['model']
if hasattr(best_wrapper.model, 'predict_proba'):
    train_sizes, train_scores, val_scores = learning_curve(
        best_wrapper.model, X, y,
        train_sizes=[0.10, 0.25, 0.50, 0.75, 1.00],
        cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE),
        scoring='roc_auc',
        n_jobs=-1,
    )

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(train_sizes, train_scores.mean(axis=1), 'o-', label='Train AUC')
    ax.fill_between(train_sizes,
                    train_scores.mean(axis=1) - train_scores.std(axis=1),
                    train_scores.mean(axis=1) + train_scores.std(axis=1), alpha=0.2)
    ax.plot(train_sizes, val_scores.mean(axis=1), 'o-', label='Val AUC')
    ax.fill_between(train_sizes,
                    val_scores.mean(axis=1) - val_scores.std(axis=1),
                    val_scores.mean(axis=1) + val_scores.std(axis=1), alpha=0.2)
    ax.set_xlabel('Training samples')
    ax.set_ylabel('AUC-ROC')
    ax.set_title(f'Learning Curves — {best_name}')
    ax.legend()
    plt.tight_layout()
    plt.savefig(str(PLOTS_DIR / 'learning_curves.png'), dpi=150)
    plt.show()
else:
    print('Learning curves skipped for neural net wrapper')

## 13. Save Best Model

In [ ]:
import joblib

best_model_path = MODELS_DIR / 'best_model.joblib'
best_wrapper.save(best_model_path)

# Save metadata
import json
metadata = {
    'best_model': best_name,
    'val_auc_roc': best_res['metrics']['auc_roc'],
    'val_auc_pr': best_res['metrics']['auc_pr'],
    'val_f1': best_res['metrics']['f1'],
    'threshold': best_res['metrics']['threshold'],
    'feature_names': list(X.columns),
    'n_features': X.shape[1],
}
with open(MODELS_DIR / 'best_model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(f'Best model ({best_name}) saved to {best_model_path}')
print(f'Val AUC-ROC: {best_res["metrics"]["auc_roc"]:.4f}')
print(f'Threshold:   {best_res["metrics"]["threshold"]:.4f}')

# Final comparison table
print('\n--- Final Model Comparison ---')
comparison_df[display_cols].round(4)